In [ ]:
import pandas as pd
import numpy as np

: 

In [ ]:
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")

In [ ]:
print("Orders shape:", orders.shape)
print("Items shape:", items.shape)
print("Customers shape:", customers.shape)

In [ ]:
print("Orders columns:")
print(orders.columns)

print("\nItems columns:")
print(items.columns)

print("\nCustomers columns:")
print(customers.columns)

In [ ]:
orders.head()

In [ ]:
items.head()

In [ ]:
customers.head()

### Data Audit for Null Values

In [ ]:
orders.isnull().sum()

In [ ]:
items.isnull().sum()

In [ ]:
customers.isnull().sum()

### Datetime related columns needed to be converted to datetime


In [ ]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])
orders.info()

### Derived columns in Orders

In [ ]:
orders['delivery_days'] = (
    orders['order_delivered_customer_date'] -
    orders['order_purchase_timestamp']
).dt.days
orders[['order_purchase_timestamp', 'order_delivered_customer_date', 'delivery_days']].head()

In [ ]:
orders['is_late'] = (
    orders['order_delivered_customer_date'] >
    orders['order_estimated_delivery_date']
)
orders[['delivery_days', 'is_late']].head()


In [ ]:
orders.to_csv("../data/processed/orders_clean.csv", index=False)

### Join Orders with Order_Items 

In [ ]:
orders_items = orders.merge(items, on='order_id', how='inner')
print("Orders shape:", orders.shape)
print("Items shape:", items.shape)
print("Merged shape:", orders_items.shape)

In [ ]:
orders_items.columns

In [ ]:
orders_items.head()

In [ ]:
orders_items.to_csv("../data/processed/orders_items_stage1.csv", index=False)

### Creating Master Table for Orders (Orders + Order_Items + Customers)

In [ ]:
master_stage1 = orders_items.merge(customers, on='customer_id', how='inner')
print("Orders + Items:", orders_items.shape)
print("Master stage 1:", master_stage1.shape)

In [ ]:
master_stage1.columns

In [ ]:
master_stage1[['order_id', 'customer_id', 'customer_city', 'customer_state']].head()

In [ ]:
master_stage1.to_csv("../data/processed/master_stage1.csv", index=False)

### Creating Master table by joining the above with products

In [ ]:
products = pd.read_csv("../data/raw/olist_products_dataset.csv")

In [ ]:
products.columns

In [ ]:
products.rename(columns={'product_name_lenght': 'product_name_length'}, inplace=True)
products.rename(columns={'product_description_lenght': 'product_description_length'}, inplace=True)

products.isnull().sum()


In [ ]:
master_stage2 = master_stage1.merge(products, on='product_id', how='left')
print("Before merge:", master_stage1.shape)
print("After merge:", master_stage2.shape)

In [ ]:
master_stage2.columns

In [ ]:
master_stage2[['product_id', 'product_category_name', 'price']].head()

In [ ]:
master_stage2[['product_category_name']].isnull().sum()

In [ ]:
master_stage2['product_category_name'] = master_stage2['product_category_name'].fillna('Unknown')

In [ ]:
master_stage2.to_csv("../data/processed/master_stage2.csv", index=False)

### Adding extra informations

In [ ]:
master_stage2['profit_margin'] = (
    (master_stage2['price'] - master_stage2['freight_value']) /
    master_stage2['price']
) * 100

In [ ]:
master_stage2[['price', 'freight_value', 'profit_margin']].head()

In [ ]:
order_counts = master_stage2.groupby('order_id').size().reset_index(name='item_count')
master_stage2 = master_stage2.merge(order_counts, on='order_id', how='left')
master_stage2[['order_id', 'item_count']].head()

In [ ]:
def classify_order_size(x):
    if x == 1:
        return 'Small'
    elif x <= 3:
        return 'Medium'
    else:
        return 'Large'

master_stage2['order_size'] = master_stage2['item_count'].apply(classify_order_size)
master_stage2[['item_count', 'order_size']].head()

In [ ]:
master_stage2['profit_margin'].describe()

In [ ]:
master_stage2.sort_values('profit_margin').head(10)[
    ['price', 'freight_value', 'profit_margin', 'product_category_name']
]

In [ ]:
import matplotlib.pyplot as plt
# SET SCALE TO BETTER SHOW
master_stage2["profit_margin"].hist(bins=100)
plt.xlim(-50,+50)
plt.title("Profit Margin Distribution")
plt.show()